# 02 — Parse Documents

## Purpose
Extracts text content from every pipeline-ready document using
`AI_PARSE_DOCUMENT` and populates two downstream tables:
`DOCUMENTS_TEXT` (one row per file, raw JSON value from Cortex) and
`DOCUMENTS_PAGES` (one row per page, plain text content ready for
translation and classification).

## What this notebook does
Queries `DOCUMENTS_INGESTED` for files with `STATUS = 'PENDING'` that
have not yet been parsed, runs `AI_PARSE_DOCUMENT` in `LAYOUT` mode on
each file in parallel (up to 4 concurrent Cortex calls), then flattens
the per-page content into `DOCUMENTS_PAGES`.

Image formats (`.jpg`, `.jpeg`, `.png`, `.tif`, `.tiff`) are parsed
without `page_split` since Cortex does not support it for those formats —
they always produce a single page row. All other formats use
`page_split: TRUE` to produce one row per page.

Only the `value` portion of the Cortex JSON response is stored —
`error` and `metadata` are discarded. Page population reads from
`DOCUMENTS_TEXT` directly so no Cortex calls are repeated if re-run.

Status is updated in `DOCUMENTS_INGESTED` to `PARSED` on success or
`PARSE_ERROR` on failure. 

## Outputs
| Table | What is written |
|---|---|
| `PROCESSING.DOCUMENTS_TEXT` | One row per file — raw value JSON, page count |
| `PROCESSING.DOCUMENTS_PAGES` | One row per page — page number, text content |
| `INGEST.DOCUMENTS_INGESTED` | STATUS updated to `PARSED` or `PARSE_ERROR` |

In [ ]:
import io
import os
import json
import pytz
import pandas as pd
from datetime import datetime, timezone
from concurrent.futures import ThreadPoolExecutor, as_completed
from snowflake.snowpark.context import get_active_session

RAW_STAGE         = '@PERMAFROST_POC.INGEST.RAW_DOCUMENTS_STAGE'
PDF_STAGE         = '@PERMAFROST_POC.PROCESSING.PROCESSED_DOC_STAGE'
DB                = 'PERMAFROST_POC'
INGEST_SCHEMA     = 'INGEST'
PROCESSING_SCHEMA = 'PROCESSING'
MAX_WORKERS       = 4

PARSEABLE_FORMATS = {'.pdf', '.docx', '.jpg', '.jpeg', '.png', '.tif', '.tiff'}

s = get_active_session() 

def info(msg):    print(f"INFO:    {msg}")
def warning(msg): print(f"WARNING: {msg}")
def error(msg):   print(f"ERROR:   {msg}")

def now_ast(): # could be replace with any time zone later
    tz = pytz.timezone('America/Halifax')
    return datetime.now(tz).isoformat()

In [ ]:
pending = s.sql(f"""
    SELECT
        i.DOC_ID,
        i.ORIGINAL_FILENAME,
        i.STAGE_PATH,
        i.SOURCE_FORMAT,
        i.SOURCE_CHANNEL
    FROM {DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED i
    LEFT JOIN {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_TEXT t
        ON i.DOC_ID = t.DOC_ID
    WHERE i.STATUS     = 'PENDING'
      AND i.SOURCE_FORMAT IN (
          '.pdf', '.docx', '.jpg', '.jpeg',
          '.png', '.tif', '.tiff'
      )
      AND t.DOC_ID IS NULL   -- not yet parsed
    ORDER BY i.RECEIVED_AT
""").collect()

parse_queue = [
    {
        'doc_id':         row['DOC_ID'],
        'filename':       row['ORIGINAL_FILENAME'],
        'stage_path':     row['STAGE_PATH'],
        'source_format':  row['SOURCE_FORMAT'],
        'source_channel': row['SOURCE_CHANNEL'],
    }
    for row in pending
]

if not parse_queue:
    print("\nNothing to parse — all PENDING files already in DOCUMENTS_TEXT "
          "or no PENDING files exist.")


In [ ]:
IMAGE_FORMATS = {'.jpg', '.jpeg', '.png', '.tif', '.tiff'}

def parse_one(f):
    doc_id, stage_path, filename = f['doc_id'], f['stage_path'], f['filename']
    ext = os.path.splitext(filename)[1].lower()

    try:

        # Safety guard — skip if already parsed in DOCUMENTS_TEXT
        if s.sql(f"""
            SELECT 1 FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_TEXT
            WHERE DOC_ID = '{doc_id}' LIMIT 1
        """).collect():
            return {'_skipped': True, 'DOC_ID': doc_id}, None

        if stage_path.startswith(RAW_STAGE):
            stage_ref, file_rel = RAW_STAGE, stage_path[len(RAW_STAGE):].lstrip('/')
        elif stage_path.startswith(PDF_STAGE):
            stage_ref, file_rel = PDF_STAGE, stage_path[len(PDF_STAGE):].lstrip('/')
        else:
            raise ValueError(f"Unrecognised stage path: {stage_path}")

        config = "{'mode': 'LAYOUT'}" if ext in IMAGE_FORMATS \
            else "{'mode': 'LAYOUT', 'page_split': TRUE}"

        raw_json = s.sql(f"""
            SELECT AI_PARSE_DOCUMENT(
                TO_FILE('{stage_ref}', '{file_rel}'),
                {config},
                TRUE
            ) AS parsed
        """).collect()[0]['PARSED']

        result = json.loads(raw_json)

        if result is None or result.get('error'):
            raise ValueError(
                result.get('error') if result else 'AI_PARSE_DOCUMENT returned NULL'
            )

        value      = result.get('value', {})
        pages      = value.get('pages') or [
            {'index': 0, 'content': value.get('content', '')}
        ]
        sorted_pages = sorted(pages, key=lambda x: x['index'])

        full_text = '\n\n'.join(
            f"[PAGE {p['index'] + 1}]\n{p.get('content', '').strip()}"
            for p in sorted_pages
            if p.get('content', '').strip()
        )
        
        page_count = len([
            p for p in pages
            if p.get('content', '').strip()
        ])

        s.sql(f"UPDATE {DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED "
              f"SET STATUS = 'PARSED' WHERE DOC_ID = '{doc_id}'").collect()

        return {
            'DOC_ID':           doc_id,
            'EXTRACTED_TEXT': full_text,
            'RAW_EXTRACTED_VALUE':   json.dumps(value),
            'PAGE_COUNT':       page_count,
            'EXTRACTION_MODEL': 'AI_PARSE_DOCUMENT',
            'EXTRACTED_AT':     now_ast(),
        }, None

    except Exception as e:
        try:
            s.sql(
                f"UPDATE {DB}.{INGEST_SCHEMA}.DOCUMENTS_INGESTED "
                f"SET STATUS = 'PARSE_ERROR' WHERE DOC_ID = '{doc_id}'"
            ).collect()
        except Exception:
            pass
        return None, {'doc_id': doc_id, 'file': filename, 'error': str(e)}


# Run parse
text_rows, parse_errors = [], []

if parse_queue:
    with ThreadPoolExecutor(max_workers=MAX_WORKERS) as executor:
        futures = {executor.submit(parse_one, f): f for f in parse_queue}
        for future in as_completed(futures):
            filename = futures[future]['filename']
            text_row, error_row = future.result()

            if text_row and text_row.get('_skipped'):
                info(f"  [SKIP] {filename} — already in DOCUMENTS_TEXT")
            elif text_row:
                text_rows.append(text_row)
                info(f"  [OK]   {filename} — {text_row['PAGE_COUNT']} page(s)")
            else:
                parse_errors.append(error_row)
                error(f"  [FAIL] {filename}: {error_row['error']}")

# Write DOCUMENTS_TEXT
if text_rows:
    s.write_pandas(
        pd.DataFrame(text_rows),
        table_name='DOCUMENTS_TEXT',
        database=DB, schema=PROCESSING_SCHEMA,
        overwrite=False,
    )
    info(f"Wrote {len(text_rows)} row(s) to DOCUMENTS_TEXT")

In [ ]:
#Read JSON output from AI_PARSE_DOCUMENT from DOCUMENTS_TEXT

rows = s.sql(f"""
    SELECT t.DOC_ID, t.RAW_EXTRACTED_VALUE
    FROM {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_TEXT t
    LEFT JOIN {DB}.{PROCESSING_SCHEMA}.DOCUMENTS_PAGES p
        ON t.DOC_ID = p.DOC_ID
    WHERE t.EXTRACTED_TEXT IS NOT NULL
      AND p.DOC_ID IS NULL   -- not yet in pages table
""").collect()

info(f"Found {len(rows)} document(s) to populate into DOCUMENTS_PAGES")


# Parse JSON and build page rows

page_rows    = []
parse_errors = []

for row in rows:
    doc_id   = row['DOC_ID']
    raw_json = row['RAW_EXTRACTED_VALUE']

    try:
        value = json.loads(raw_json)

        if value is None:
            raise ValueError('NULL result in RAW_EXTRACTED_VALUE')

        # PDF with page_split → pages array
        # Image / DOCX fallback → top-level content string
        pages = value.get('pages') or [
            {'index': 0, 'content': value.get('content', '')}
        ]

        doc_pages = [
            {
                'DOC_ID':                  doc_id,
                'PAGE_INDEX':              p['index'],
                'PAGE_NUMBER':             p['index'] + 1,
                'PAGE_CONTENT':            p.get('content', '').strip(),
                'PAGE_CONTENT_TRANSLATED': None,
            }
            for p in sorted(pages, key=lambda x: x['index'])
            if p.get('content', '').strip()
        ]

        page_rows.extend(doc_pages)
        info(f"  [OK] {doc_id} — {len(doc_pages)} page(s)")

    except Exception as e:
        parse_errors.append({'doc_id': doc_id, 'error': str(e)})
        error(f"  [FAIL] {doc_id}: {e}")


# write to DOCUMENTS_PAGES

if page_rows:
    s.write_pandas(
        pd.DataFrame(page_rows),
        table_name='DOCUMENTS_PAGES',
        database=DB,
        schema=PROCESSING_SCHEMA,
        overwrite=False,
    )
    info(f"Wrote {len(page_rows)} row(s) to DOCUMENTS_PAGES")
else:
    info("No page rows to write")